In [ ]:
!pip install torch torchvision timm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# IMPORTS

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from torchvision import datasets, models
from torch.utils.data import DataLoader, Subset
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import os
import time

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("PyTorch version:", torch.__version__)

Device: cuda
PyTorch version: 2.11.0+cu130


# Downloading cifar10

In [ ]:

import os
import tarfile
from torchvision import datasets, transforms

# ImageNet normalization for pretrained MobileNetV2
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

# Training transformations
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Testing transformations
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load CIFAR-10 from Google Drive
data_path = "/content/drive/MyDrive/ECE662 Final Project"
archive = os.path.join(data_path, "cifar-10-python.tar.gz")

# Extract dataset only if necessary
extracted_path = os.path.join(data_path, "cifar-10-batches-py")

if not os.path.exists(extracted_path):
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path=data_path, filter="data")

# Load training dataset
train_data = datasets.CIFAR10(
    root=data_path,
    train=True,
    download=False,
    transform=train_transform
)

# Load testing dataset
test_data = datasets.CIFAR10(
    root=data_path,
    train=False,
    download=False,
    transform=test_transform
)

print("Training images:", len(train_data))
print("Testing images:", len(test_data))
print("Classes:", train_data.classes)


100%|██████████| 170M/170M [30:50<00:00, 92.1kB/s]


Training images: 50000
Testing images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# Creating data batches

In [ ]:
batch_size = 128

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# Create a reproducible 1,000-image test subset
generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(test_data),
    generator=generator
)[:1000].tolist()

fast_test_data = Subset(test_data, indices)

fast_loader = DataLoader(
    fast_test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("Full test set:", len(test_data))
print("Fast test set:", len(fast_test_data))

Full test set: 10000
Fast test set: 1000


# Load pretrained MobileNetV2

In [ ]:
from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights
)

# Download pretrained ImageNet model
model = mobilenet_v2(
    weights=MobileNet_V2_Weights.DEFAULT
)

# Replace the classifier for CIFAR-10
# ImageNet: 1000 classes
# CIFAR-10: 10 classes

model.classifier[1] = nn.Linear(
    model.last_channel,
    10
)

model = model.to(device)

print("MobileNetV2 loaded successfully!")
print("Output classes:", model.classifier[1].out_features)

MobileNetV2 loaded successfully!
Output classes: 10


# Eval function

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    return accuracy

print("Evaluation function ready!")

Evaluation function ready!


# Fine Tuning

In [ ]:
# Separate validation data with no random augmentation
val_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=test_transform
)

# Split training data into 45,000 training
# and 5,000 validation images
generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(
    len(train_data),
    generator=generator
).tolist()

train_indices = permutation[:45000]
val_indices = permutation[45000:]

train_subset = Subset(train_data, train_indices)
val_subset = Subset(val_data, val_indices)

train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("Training:", len(train_subset))
print("Validation:", len(val_subset))

Training: 45000
Validation: 5000


In [ ]:
# num_epochs = 10
# criterion = nn.CrossEntropyLoss()

# optimizer = optim.AdamW([
#     {
#         "params": model.features.parameters(),
#         "lr": 0.0001
#     },
#     {
#         "params": model.classifier.parameters(),
#         "lr": 0.001
#     }
# ], weight_decay=0.0001)

# scheduler = optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max=num_epochs
# )

# best_accuracy = 0.0
# checkpoint_path = "/content/drive/MyDrive/mobilenetv2_cifar10.pth"

# for epoch in range(num_epochs):

#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     start_time = time.time()

#     for images, labels in train_loader:

#         images = images.to(device)
#         labels = labels.to(device)

#         optimizer.zero_grad(set_to_none=True)

#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#         predictions = outputs.argmax(dim=1)
#         correct += (predictions == labels).sum().item()
#         total += labels.size(0)

#     scheduler.step()

#     train_acc = 100 * correct / total
#     val_acc = evaluate(model, val_loader)

#     if val_acc > best_accuracy:
#         best_accuracy = val_acc

#         torch.save(
#             model.state_dict(),
#             checkpoint_path
#         )

#         print("Best model saved!")

#     print(
#         f"Epoch {epoch+1}/{num_epochs} | "
#         f"Train: {train_acc:.2f}% | "
#         f"Validation: {val_acc:.2f}% | "
#         f"Time: {time.time()-start_time:.1f}s"
#     )

# print("Best validation accuracy:", best_accuracy)

# Saving and reload if necesary

In [ ]:
model = mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(
    model.last_channel, 10
)

state = torch.load(
    "/content/drive/MyDrive/ECE662 Final Project/mobilenetv2_cifar10.pth",
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state)
model = model.to(device).eval()

# Accuracy on fine tuned model

In [ ]:
# Fast accuracy evaluation
fast_accuracy = evaluate(
    model,
    fast_loader
)

# Full test accuracy evaluation
fp32_accuracy = evaluate(
    model,
    test_loader
)

print(f"Fast accuracy: {fast_accuracy:.2f}%")
print(f"Full accuracy: {fp32_accuracy:.2f}%")

Fast accuracy: 89.00%
Full accuracy: 89.82%


# Quantize weights

In [ ]:
def quantize_weights(weight, bits):
    qmax = 2 ** (bits - 1) - 1

    # Per-output-channel scaling
    dims = tuple(range(1, weight.ndim))

    max_value = weight.abs().amax(
        dim=dims, keepdim=True
    )

    scale = max_value.clamp(min=1e-8) / qmax

    q_weight = torch.round(weight / scale)
    q_weight = torch.clamp(q_weight, -qmax, qmax)

    return q_weight * scale


def quantize_model(original_model, bits):
    quant_model = copy.deepcopy(original_model)

    with torch.no_grad():
        for layer in quant_model.modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                layer.weight.copy_(
                    quantize_weights(layer.weight, bits)
                )

    return quant_model.to(device).eval()

In [ ]:
model_int8 = quantize_model(model, 8)
model_int4 = quantize_model(model, 4)

print("INT8 and INT4 models created!")

INT8 and INT4 models created!


In [ ]:
int8_accuracy = evaluate(model_int8, test_loader)
int4_accuracy = evaluate(model_int4, test_loader)

print(f"FP32 Accuracy: {fp32_accuracy:.2f}%")
print(f"INT8 Accuracy: {int8_accuracy:.2f}%")
print(f"INT4 Accuracy: {int4_accuracy:.2f}%")

FP32 Accuracy: 89.82%
INT8 Accuracy: 89.87%
INT4 Accuracy: 17.58%


# model sizes

In [ ]:
def model_size_mb(model, bits=None):
    if bits is None:
        total_bytes = sum(
            p.numel() * p.element_size()
            for p in model.parameters()
        )
    else:
        total_bytes = 0

        for layer in model.modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                total_bytes += layer.weight.numel() * bits / 8
                total_bytes += layer.weight.shape[0] * 4

                if layer.bias is not None:
                    total_bytes += layer.bias.numel() * 4

            elif isinstance(layer, nn.BatchNorm2d):
                if layer.affine:
                    total_bytes += (
                        layer.weight.numel()
                        + layer.bias.numel()
                    ) * 4

    return total_bytes / (1024 ** 2)


fp32_size = model_size_mb(model)
int8_size = model_size_mb(model, 8)
int4_size = model_size_mb(model, 4)

# Results

In [ ]:
results = pd.DataFrame({
    "Model": ["FP32", "INT8", "INT4"],
    "Accuracy (%)": [
        fp32_accuracy,
        int8_accuracy,
        int4_accuracy
    ],
    "Accuracy Drop (pp)": [
        0,
        fp32_accuracy - int8_accuracy,
        fp32_accuracy - int4_accuracy
    ],
    "Estimated Size (MiB)": [
        fp32_size,
        int8_size,
        int4_size
    ]
})

display(results.round(2))

results.to_csv(
    "/content/quantization_results.csv",
    index=False
)

,Model,Accuracy (%),Accuracy Drop (pp),Estimated Size (MiB)
0,FP32,89.82,0.00,8.53
1,INT8,89.87,-0.05,2.30
2,INT4,17.58,72.24,1.25
